# Transformação de Boletos é Auxiliar com PySpark
This notebook uses **PySpark** to transform, inspect, and explore `base_boletos_fiap.csv and base_auxiliar.csv`.

## 1. Imports & SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from delta import configure_spark_with_delta_pip
from datetime import datetime

builder = (
    SparkSession.builder
    .appName('Tabelas Gold')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

Spark version: 4.1.1


## 2. Read from Silver
Load the raw Delta table written by the silver notebook.

In [2]:
SILVER_AUXILIAR    = "../../output_data/silver/base_auxiliar"
SILVER_BOLETOS  = "../../output_data/silver/boletos"
GOLD   = "../../output_data/gold"

silver_boletos = spark.read.format("delta").load(SILVER_BOLETOS)
silver_aux = spark.read.format("delta").load(SILVER_AUXILIAR)

In [3]:
silver_boletos.printSchema()

root
 |-- id_boleto: string (nullable = true)
 |-- id_pagador: string (nullable = true)
 |-- id_beneficiario: string (nullable = true)
 |-- dt_emissao: date (nullable = true)
 |-- dt_vencimento: date (nullable = true)
 |-- dt_pagamento: date (nullable = true)
 |-- vlr_nominal: double (nullable = true)
 |-- vlr_baixa: double (nullable = true)
 |-- tipo_baixa: string (nullable = true)
 |-- tipo_especie: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- partition_date: date (nullable = true)
 |-- boleto_pago: boolean (nullable = true)
 |-- dias_ate_vencimento: integer (nullable = true)
 |-- dias_atraso: integer (nullable = true)
 |-- boleto_pago_em_atraso: boolean (nullable = true)
 |-- vlr_diferenca: double (nullable = true)
 |-- mes_emissao: string (nullable = true)
 |-- categoria_baixa: string (nullable = true)
 |-- faixa_valor: string (nullable = true)
 |-- ano_mes_emissao: string (nullable = true)
 |-- ano_mes_vencimento: string (nullable = true)



In [4]:
silver_aux.printSchema()

root
 |-- id_cnpj: string (nullable = true)
 |-- cd_cnae_prin: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- sacado_indice_liquidez_1m: string (nullable = true)
 |-- cedente_indice_liquidez_1m: string (nullable = true)
 |-- score_materialidade_evolucao: string (nullable = true)
 |-- media_atraso_dias: double (nullable = true)
 |-- indicador_liquidez_quantitativo_3m: double (nullable = true)
 |-- share_vl_inad_pag_bol_6_a_15d: string (nullable = true)
 |-- score_quantidade_v2: string (nullable = true)
 |-- score_materialidade_v2: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- partition_date: date (nullable = true)
 |-- grupo_cnae: string (nullable = true)
 |-- setor_economico: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- faixa_liquidez_sacado: string (nullable = true)
 |-- faixa_score_materialidade: string (nullable = true)
 |-- flag_inadimplencia_alta: boolean (nullable = true)



In [5]:
cols_auxiliar= [
    "id_cnpj",
    "cd_cnae_prin",
    "uf",
    "sacado_indice_liquidez_1m",
    "cedente_indice_liquidez_1m",
    "score_materialidade_evolucao",
    "media_atraso_dias",
    "indicador_liquidez_quantitativo_3m",
    "share_vl_inad_pag_bol_6_a_15d",
    "score_quantidade_v2",
    "score_materialidade_v2",
    "grupo_cnae",
    "setor_economico",
    "regiao",
    "faixa_liquidez_sacado",
    "faixa_score_materialidade",
    "flag_inadimplencia_alta",
]

df_silver_enriched = (
    silver_boletos.alias("a")
    .join(
        silver_aux.alias("b"),
        F.col("a.id_pagador") == F.col("b.id_cnpj"),
        "left"
    ).select(
        "a.*",
        *cols_auxiliar,
    ).drop(
        F.col("partition_date")
    )   
    
    
    .withColumn("tem_perfil_pagador",
        F.when(F.col("id_cnpj").isNotNull(), True).otherwise(False))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn('partition_date', F.current_date())
)

df_silver_enriched.printSchema()

root
 |-- id_boleto: string (nullable = true)
 |-- id_pagador: string (nullable = true)
 |-- id_beneficiario: string (nullable = true)
 |-- dt_emissao: date (nullable = true)
 |-- dt_vencimento: date (nullable = true)
 |-- dt_pagamento: date (nullable = true)
 |-- vlr_nominal: double (nullable = true)
 |-- vlr_baixa: double (nullable = true)
 |-- tipo_baixa: string (nullable = true)
 |-- tipo_especie: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- boleto_pago: boolean (nullable = true)
 |-- dias_ate_vencimento: integer (nullable = true)
 |-- dias_atraso: integer (nullable = true)
 |-- boleto_pago_em_atraso: boolean (nullable = true)
 |-- vlr_diferenca: double (nullable = true)
 |-- mes_emissao: string (nullable = true)
 |-- categoria_baixa: string (nullable = true)
 |-- faixa_valor: string (nullable = true)
 |-- ano_mes_emissao: string (nullable = true)
 |-- ano_mes_vencimento: string (nullable = true)
 |-- id_cnpj: string (nullable = true)
 |-- cd

In [6]:
df_silver_enriched.show(100, truncate = False)

+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------+-------------+------------+-----------+---------+--------------------------------------------------------------+---------------------------------+--------------------------+-----------+-------------------+-----------+---------------------+-------------+-----------+-----------------------+-----------+---------------+------------------+----------------------------------------------------------------+------------+----+-------------------------+--------------------------+----------------------------+------------------+----------------------------------+-----------------------------+-------------------+----------------------+----------+---------------+-------------+---------------------+-------------------------+-----------------------+------------------+--------------+
|id_boleto       

## 3. Write to gold
Persist the enriched DataFrame as a Delta table.

In [7]:
df_silver_enriched.write.format("delta").mode("overwrite").partitionBy('partition_date') \
    .option("overwriteSchema", "true").save(GOLD)

total = df_silver_enriched.count()
com_perfil = df_silver_enriched.filter(F.col("tem_perfil_pagador") == True).count()

print(f" silver_boletos_enriquecido: {total:,} registros | {len(df_silver_enriched.columns)} colunas")
print(f"   Match com perfil pagador: {com_perfil:,}/{total:,} ({100*com_perfil/total:.1f}%)")
display(df_silver_enriched.limit(5))

 silver_boletos_enriquecido: 7,118 registros | 40 colunas
   Match com perfil pagador: 7,118/7,118 (100.0%)


DataFrame[id_boleto: string, id_pagador: string, id_beneficiario: string, dt_emissao: date, dt_vencimento: date, dt_pagamento: date, vlr_nominal: double, vlr_baixa: double, tipo_baixa: string, tipo_especie: string, ingestion_timestamp: timestamp, boleto_pago: boolean, dias_ate_vencimento: int, dias_atraso: int, boleto_pago_em_atraso: boolean, vlr_diferenca: double, mes_emissao: string, categoria_baixa: string, faixa_valor: string, ano_mes_emissao: string, ano_mes_vencimento: string, id_cnpj: string, cd_cnae_prin: string, uf: string, sacado_indice_liquidez_1m: string, cedente_indice_liquidez_1m: string, score_materialidade_evolucao: string, media_atraso_dias: double, indicador_liquidez_quantitativo_3m: double, share_vl_inad_pag_bol_6_a_15d: string, score_quantidade_v2: string, score_materialidade_v2: string, grupo_cnae: string, setor_economico: string, regiao: string, faixa_liquidez_sacado: string, faixa_score_materialidade: string, flag_inadimplencia_alta: boolean, tem_perfil_pagador: 

## 4. Write to gold to csv
Because  .

In [8]:
today_date = datetime.now().strftime("%Y-%m-%d")
df_silver_enriched.write.csv(f'../../output_data/gold/dataviz/{today_date}', header= True)